# E-commerce Sales Analysis — Step 1: Data Cleaning

**Project:** E-commerce Sales & Customer Retention Analysis
**Dataset:** `ecommerce_sales_raw.csv` (raw export, ~50,750 rows)
**Goal of this notebook:** take the messy raw export and turn it into a clean dataset (`ecommerce_sales_clean.csv`) that we can safely use for EDA, SQL and Power BI.

We'll go step by step, and print the results after each step so you can see exactly what changed and why.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

df = pd.read_csv('ecommerce_sales_raw.csv')
print("Shape (rows, columns):", df.shape)
df.head()

Shape (rows, columns): (50750, 12)


,OrderID,CustomerID,OrderDate,City,ProductCategory,ProductName,Quantity,UnitPrice,DiscountPercent,PaymentMethod,OrderStatus,CustomerType
0,ORD011764,CUST03063,2023-02-15,Bhopal,Fashion,Sports Cap,1,5661.44,0.0,Credit Card,Delivered,New
1,ORD031296,CUST03595,28-Jun-2023,NaN,Electronics,Keyboard,3,3014.89,5.0,Debit Card,Returned,Returning
2,ORD030454,CUST06563,15-Jan-2024,Kolkata,Home & Kitchen,Cutlery Set,2,872.64,25.0,UPI,Delivered,Returning
3,ORD021845,CUST06696,2023-11-16,Delhi,Home & Kitchen,Bedsheet Set,2,6383.86,20.0,UPI,Delivered,New
4,ORD015679,CUST01568,30/04/2023,Bengaluru,Sports,Cycling Helmet,2,4391.31,20.0,Debit Card,Delivered,Returning


## Step 1: Initial Inspection

Before touching anything, let's understand what we're working with — data types, missing values, and duplicates.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50750 entries, 0 to 50749
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          50750 non-null  str    
 1   CustomerID       50750 non-null  str    
 2   OrderDate        50750 non-null  str    
 3   City             46347 non-null  str    
 4   ProductCategory  50750 non-null  str    
 5   ProductName      50750 non-null  str    
 6   Quantity         50750 non-null  int64  
 7   UnitPrice        50238 non-null  str    
 8   DiscountPercent  49739 non-null  float64
 9   PaymentMethod    42179 non-null  str    
 10  OrderStatus      50750 non-null  str    
 11  CustomerType     49229 non-null  str    
dtypes: float64(1), int64(1), str(10)
memory usage: 4.6 MB


In [3]:
print("Missing values per column:\n")
print(df.isna().sum())

print("\nDuplicate rows (excluding OrderID, since IDs are always unique even for a copy-paste error):")
dup_check_cols = [c for c in df.columns if c != 'OrderID']
print(df.duplicated(subset=dup_check_cols).sum())

Missing values per column:

OrderID               0
CustomerID            0
OrderDate             0
City               4403
ProductCategory       0
ProductName           0
Quantity              0
UnitPrice           512
DiscountPercent    1011
PaymentMethod      8571
OrderStatus           0
CustomerType       1521
dtype: int64

Duplicate rows (excluding OrderID, since IDs are always unique even for a copy-paste error):
662


## Step 2: Remove Duplicate Rows

Raw exports often contain the same order pasted in twice. We check duplicates on every column *except* `OrderID` (since a true duplicate row will differ only by having two different generated IDs — but here since we generated dupes as pure repeats, we drop full-row duplicates too).

In [4]:
before = len(df)
df = df.drop_duplicates(subset=dup_check_cols, keep='first').reset_index(drop=True)
after = len(df)
print(f"Removed {before - after} duplicate rows. New shape: {df.shape}")

Removed 662 duplicate rows. New shape: (50088, 12)


## Step 3: Fix `OrderDate` — Mixed Formats

The raw export has dates in at least 4 different formats (`YYYY-MM-DD`, `DD/MM/YYYY`, `DD-Mon-YYYY`, `MM/DD/YYYY`). We can't parse them all with a single format string, so we try several known formats in order until one works for each row.

In [5]:
def parse_mixed_date(value):
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%d-%b-%Y", "%m/%d/%Y"]
    for fmt in formats:
        try:
            return pd.to_datetime(value, format=fmt)
        except (ValueError, TypeError):
            continue
    return pd.NaT

df['OrderDate'] = df['OrderDate'].apply(parse_mixed_date)

print("Unparseable dates remaining:", df['OrderDate'].isna().sum())
print("Date range:", df['OrderDate'].min(), "to", df['OrderDate'].max())

Unparseable dates remaining: 0
Date range: 2023-01-01 00:00:00 to 2024-12-31 00:00:00


## Step 4: Fix `UnitPrice` — Text Values Like "Rs.2217.53"

Some rows have the price stored as text with a currency prefix instead of a number. We strip the prefix and convert everything to a proper float column.

In [6]:
def clean_price(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        value = value.replace('Rs.', '').replace('Rs', '').strip()
        try:
            return float(value)
        except ValueError:
            return np.nan
    return float(value)

df['UnitPrice'] = df['UnitPrice'].apply(clean_price)
print("UnitPrice dtype now:", df['UnitPrice'].dtype)
print("Missing UnitPrice values:", df['UnitPrice'].isna().sum())

UnitPrice dtype now: float64
Missing UnitPrice values: 502


## Step 5: Fix Negative `Quantity` Values

A small number of rows have negative quantities — a data-entry error in the raw export (they should never be negative; a real return would show up in `OrderStatus`, not as a negative count). We convert these to their absolute value.

In [7]:
neg_count = (df['Quantity'] < 0).sum()
df['Quantity'] = df['Quantity'].abs()
print(f"Fixed {neg_count} negative Quantity values by converting to positive.")

Fixed 254 negative Quantity values by converting to positive.


## Step 6: Standardize Categorical Text

`ProductCategory` sometimes appears in ALL CAPS, and `City` has inconsistent casing (e.g. "Bangalore" vs "Bengaluru" vs "surat"). We standardize casing and merge known duplicate city names.

In [8]:
df['ProductCategory'] = df['ProductCategory'].str.title().str.strip()

df['City'] = df['City'].str.strip().str.title()
df['City'] = df['City'].replace({'Bangalore': 'Bengaluru'})

df['CustomerType'] = df['CustomerType'].astype(str).str.strip().str.title().replace('Nan', np.nan)
df['PaymentMethod'] = df['PaymentMethod'].replace({'credit card': 'Credit Card', 'COD': 'Cash On Delivery',
                                                      'Cash on Delivery': 'Cash On Delivery'})

print(df['ProductCategory'].unique())
print(df['City'].unique())
print(df['CustomerType'].unique())
print(df['PaymentMethod'].unique())

<StringArray>
['Fashion', 'Electronics', 'Home & Kitchen', 'Sports', 'Books', 'Beauty']
Length: 6, dtype: str
<StringArray>
[    'Bhopal',          nan,    'Kolkata',      'Delhi',  'Bengaluru',
      'Patna',    'Lucknow',     'Indore',     'Mumbai',      'Surat',
  'Hyderabad',     'Nagpur',    'Chennai',       'Pune',  'Ahmedabad',
     'Jaipur', 'Coimbatore',      'Kochi',   'Vadodara']
Length: 19, dtype: str
<StringArray>
['New', 'Returning', nan]
Length: 3, dtype: str
<StringArray>
['Credit Card', 'Debit Card', 'UPI', 'Cash On Delivery', nan, 'Net Banking']
Length: 6, dtype: str


## Step 7: Handle Missing Values

Different columns need different treatment:
- `City`, `PaymentMethod`, `CustomerType` → fill with `"Unknown"` (we don't want to drop real orders just because one field is missing)
- `DiscountPercent` → missing almost certainly means no discount was applied → fill with `0`
- `UnitPrice` → fill with the median price *for that product*, since price is fairly consistent per product

In [9]:
df['City'] = df['City'].fillna('Unknown')
df['PaymentMethod'] = df['PaymentMethod'].fillna('Unknown')
df['CustomerType'] = df['CustomerType'].fillna('Unknown')
df['DiscountPercent'] = df['DiscountPercent'].fillna(0)

df['UnitPrice'] = df.groupby('ProductName')['UnitPrice'].transform(lambda x: x.fillna(x.median()))

print("Remaining missing values:\n")
print(df.isna().sum())

Remaining missing values:

OrderID            0
CustomerID         0
OrderDate          0
City               0
ProductCategory    0
ProductName        0
Quantity           0
UnitPrice          0
DiscountPercent    0
PaymentMethod      0
OrderStatus        0
CustomerType       0
dtype: int64


## Step 8: Feature Engineering

We add a few derived columns that will make EDA, SQL and Power BI much easier:
- `GrossAmount` = Quantity × UnitPrice
- `NetAmount` = GrossAmount after discount
- `OrderYear`, `OrderMonth`, `OrderMonthName` for time-based analysis

In [10]:
df['GrossAmount'] = (df['Quantity'] * df['UnitPrice']).round(2)
df['NetAmount'] = (df['GrossAmount'] * (1 - df['DiscountPercent'] / 100)).round(2)

df['OrderYear'] = df['OrderDate'].dt.year
df['OrderMonth'] = df['OrderDate'].dt.month
df['OrderMonthName'] = df['OrderDate'].dt.strftime('%b-%Y')

df.head()

,OrderID,CustomerID,OrderDate,City,ProductCategory,ProductName,Quantity,UnitPrice,DiscountPercent,PaymentMethod,OrderStatus,CustomerType,GrossAmount,NetAmount,OrderYear,OrderMonth,OrderMonthName
0,ORD011764,CUST03063,2023-02-15,Bhopal,Fashion,Sports Cap,1,5661.44,0.0,Credit Card,Delivered,New,5661.44,5661.44,2023,2,Feb-2023
1,ORD031296,CUST03595,2023-06-28,Unknown,Electronics,Keyboard,3,3014.89,5.0,Debit Card,Returned,Returning,9044.67,8592.44,2023,6,Jun-2023
2,ORD030454,CUST06563,2024-01-15,Kolkata,Home & Kitchen,Cutlery Set,2,872.64,25.0,UPI,Delivered,Returning,1745.28,1308.96,2024,1,Jan-2024
3,ORD021845,CUST06696,2023-11-16,Delhi,Home & Kitchen,Bedsheet Set,2,6383.86,20.0,UPI,Delivered,New,12767.72,10214.18,2023,11,Nov-2023
4,ORD015679,CUST01568,2023-04-30,Bengaluru,Sports,Cycling Helmet,2,4391.31,20.0,Debit Card,Delivered,Returning,8782.62,7026.10,2023,4,Apr-2023


## Step 9: Final Checks & Save Cleaned Data

In [11]:
print("Final shape:", df.shape)
print("\nData types:\n", df.dtypes)
print("\nAny remaining missing values:", df.isna().sum().sum())

df.to_csv('ecommerce_sales_clean.csv', index=False)
print("\nSaved cleaned dataset to ecommerce_sales_clean.csv")

Final shape: (50088, 17)

Data types:
 OrderID                       str
CustomerID                    str
OrderDate          datetime64[us]
City                          str
ProductCategory               str
ProductName                   str
Quantity                    int64
UnitPrice                 float64
DiscountPercent           float64
PaymentMethod                 str
OrderStatus                   str
CustomerType                  str
GrossAmount               float64
NetAmount                 float64
OrderYear                   int32
OrderMonth                  int32
OrderMonthName                str
dtype: object

Any remaining missing values: 0

Saved cleaned dataset to ecommerce_sales_clean.csv
